# 1. Data Collecting

In [ ]:
!pip install kaggle -q

import os, shutil
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

# Setup Kaggle API key
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# Download dataset kaggle
!kaggle datasets download -d arunrk7/surface-crack-detection --unzip -p data/

Dataset URL: https://www.kaggle.com/datasets/arunrk7/surface-crack-detection
License(s): copyright-authors




  0%|          | 0.00/233M [00:00<?, ?B/s]
  0%|          | 1.00M/233M [00:01<05:02, 804kB/s]
  1%|          | 2.00M/233M [00:01<02:48, 1.44MB/s]
  1%|▏         | 3.00M/233M [00:01<01:50, 2.18MB/s]
  2%|▏         | 4.00M/233M [00:01<01:20, 3.00MB/s]
  3%|▎         | 6.00M/233M [00:02<00:48, 4.91MB/s]
  3%|▎         | 8.00M/233M [00:02<00:35, 6.72MB/s]
  4%|▍         | 10.0M/233M [00:02<00:28, 8.11MB/s]
  5%|▌         | 12.0M/233M [00:02<00:25, 9.23MB/s]
  6%|▌         | 14.0M/233M [00:02<00:22, 10.0MB/s]
  7%|▋         | 16.0M/233M [00:02<00:21, 10.6MB/s]
  8%|▊         | 18.0M/233M [00:03<00:20, 11.1MB/s]
  9%|▊         | 20.0M/233M [00:03<00:19, 11.4MB/s]
  9%|▉         | 22.0M/233M [00:03<00:18, 11.7MB/s]
 10%|█         | 24.0M/233M [00:03<00:18, 11.8MB/s]
 11%|█         | 26.0M/233M [00:03<00:18, 11.9MB/s]
 12%|█▏        | 28.0M/233M [00:04<00:17, 12.0MB/s]
 13%|█▎        | 30.0M/233M [00:04<00:17, 12.1MB/s]
 14%|█▎        | 32.0M/233M [00:04<00:17, 12.1MB/s]
 15%|█▍        | 34.0

In [ ]:
SEED = 42
CLASS_MAP = {'Negative': 0, 'Positive': 1}

paths, labels = [], []
for cls, lbl in CLASS_MAP.items():
    for f in os.listdir(f'data/{cls}'):
        paths.append(f'data/{cls}/{f}')
        labels.append(lbl)

# Split ke 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

# Simpan ke CSV 
os.makedirs('data/splits', exist_ok=True)
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    pd.DataFrame({'path': X, 'label': y}).to_csv(f'data/splits/{name}.csv', index=False)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 28000 | Val: 6000 | Test: 6000


# 2. Pre-Processing
Dalam pre-processing ini kita melakukan:
1. Mengubah gambar menjadi grayscale
2. Resize ke 128x128
3. Normalisasi rentang 0 - 1
4. Augmentasi (Flip, Rotasi, Brighness, Contrass)

In [ ]:
IMG_SIZE   = 128
BATCH_SIZE = 32

train_transform = T.Compose([
    T.Grayscale(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2),
    T.ToTensor(),
    T.Lambda(lambda x: x + 0.02 * torch.randn_like(x))  # gaussian noise
])

val_test_transform = T.Compose([
    T.Grayscale(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor()
])


class CrackDataset(Dataset):
    def __init__(self, csv_path, transform):
        df = pd.read_csv(csv_path)
        self.paths, self.labels = df['path'].tolist(), df['label'].tolist()
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.labels[idx], dtype=torch.long)


train_loader = DataLoader(CrackDataset('data/splits/train.csv', train_transform),    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(CrackDataset('data/splits/val.csv',   val_test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(CrackDataset('data/splits/test.csv',  val_test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Verifikasi
imgs, lbls = next(iter(train_loader))
print(f'Shape: {imgs.shape} | Range: [{imgs.min():.2f}, {imgs.max():.2f}] | Labels: {lbls[:8].tolist()}')